In [ ]:
#related d2l packages need to be download
#github URL: https://github.com/MockingBird-OvO/Kagggle-House-Price-Prediction.git

In [ ]:
%matplotlib inline
import pandas as pd
import torch
from torch import nn
from d2l import torch as d2l
from torch import nn

In [ ]:
#Download Data
def download(url, folder, sha1_hash=None):
    """Download a file to folder and return the local filepath."""

def extract(filename, folder):
    """Extract a zip/tar file into folder."""

In [ ]:
#Data extraction
class KaggleHouse(d2l.DataModule):
    def __init__(self, batch_size, train=None, val=None):
        super().__init__()
        self.save_hyperparameters()
        if self.train is None:
            self.raw_train = pd.read_csv(d2l.download(
                d2l.DATA_URL + 'kaggle_house_pred_train.csv', self.root,
                sha1_hash='585e9cc93e70b39160e7921475f9bcd7d31219ce'))
            self.raw_val = pd.read_csv(d2l.download(
                d2l.DATA_URL + 'kaggle_house_pred_test.csv', self.root,
                sha1_hash='fa19780a7b011d9b009e8bff8e99922a8ee2eb90'))

In [ ]:
#Verification
data = KaggleHouse(batch_size=64)
print(data.raw_train.shape)
print(data.raw_val.shape)

In [ ]:
#Preprocess data
@d2l.add_to_class(KaggleHouse)
def preprocess(self):
    # Remove the ID and label columns
    label = 'SalePrice'
    features = pd.concat(
        (self.raw_train.drop(columns=['Id', label]),
         self.raw_val.drop(columns=['Id'])))
    # Standardize numerical columns
    numeric_features = features.dtypes[features.dtypes!='object'].index
    features[numeric_features] = features[numeric_features].apply(
        lambda x: (x - x.mean()) / (x.std()))
    # Replace NAN numerical features by 0
    features[numeric_features] = features[numeric_features].fillna(0)
    # Replace discrete features by one-hot encoding
    features = pd.get_dummies(features, dummy_na=True)
    # Save preprocessed features
    self.train = features[:self.raw_train.shape[0]].copy()
    self.train[label] = self.raw_train[label]
    self.val = features[self.raw_train.shape[0]:].copy()

In [ ]:
#Example
data.preprocess()
data.train.shape, data.train.iloc[0:2, 0:3]

In [ ]:
#Change data type
@d2l.add_to_class(KaggleHouse)
def get_dataloader(self, train):
    label = 'SalePrice'
    data = self.train if train else self.val
    if label not in data: return
    get_tensor = lambda x: torch.tensor(x.values.astype(float),
                                      dtype=torch.float32)
    # Logarithm of prices
    tensors = (get_tensor(data.drop(columns=[label])),  # X
               torch.log(get_tensor(data[label])).reshape((-1, 1)))  # Y
    return self.get_tensorloader(tensors, train)

In [ ]:
#K-fold cross-validation
def k_fold_data(data, k):
    rets = []
    fold_size = data.train.shape[0] // k
    for j in range(k):
        idx = range(j * fold_size, (j+1) * fold_size)
        rets.append(KaggleHouse(data.batch_size, data.train.drop(index=idx),
                                data.train.loc[idx]))
    return rets

In [ ]:
#Model
class ImprovedMLP(d2l.Module):
    def __init__(self, num_outputs, num_hiddens_1, num_hiddens_2, num_hiddens_3, num_hiddens_4, num_hiddens_5,
                 dropout_1, dropout_2, dropout_3, dropout_4, dropout_5, lr, wd, activation='relu'):
        super().__init__()
        self.save_hyperparameters()
        
        # Choose activation function
        if activation == 'relu':
            act = nn.ReLU()
        elif activation == 'leaky_relu':
            act = nn.LeakyReLU(0.1)
        elif activation == 'elu':
            act = nn.ELU()
        elif activation == 'silu':  # Swish activation
            act = nn.SiLU()
        elif activation == 'tanh':
            act = nn.Tanh()
        else:
            act = nn.ReLU()
        
        # Build deeper network with batch normalization
        self.net = nn.Sequential(
            nn.LazyLinear(num_hiddens_1),
            nn.BatchNorm1d(num_hiddens_1),  # Batch normalization
            act,
            nn.Dropout(dropout_1),
            
            nn.LazyLinear(num_hiddens_2),
            nn.BatchNorm1d(num_hiddens_2),
            act,
            nn.Dropout(dropout_2),
            
            nn.LazyLinear(num_hiddens_3),
            nn.BatchNorm1d(num_hiddens_3),
            act,
            nn.Dropout(dropout_3),
            
            nn.LazyLinear(num_hiddens_4),
            nn.BatchNorm1d(num_hiddens_4),
            act,
            nn.Dropout(dropout_4),

            nn.LazyLinear(num_hiddens_5),
            nn.BatchNorm1d(num_hiddens_5),
            act,
            nn.Dropout(dropout_5),
            
            nn.LazyLinear(num_outputs)
        )
        
    def forward(self, X):
        return self.net(X)
    
    def loss(self, y_hat, y):
        # MSE loss for regression
        return nn.MSELoss()(y_hat, y)
    
    def configure_optimizers(self):
        # CORRECT: Apply weight decay to ALL parameters (except biases and BN)
        decay_params = []
        no_decay_params = []
        
        for name, param in self.named_parameters():
            if 'bias' in name or 'bn' in name or 'BatchNorm' in name:
                no_decay_params.append(param)
            else:
                decay_params.append(param)
        
        return torch.optim.Adam([
            {'params': decay_params, 'weight_decay': self.wd},
            {'params': no_decay_params, 'weight_decay': 0}
        ], lr=self.lr)

In [ ]:
#Training
def k_fold(trainer, data, k, hparams):
    val_loss, models = [], []
    for i, data_fold in enumerate(k_fold_data(data, k)):
        model = ImprovedMLP(**hparams)
        model.board.yscale='log'
        if i != 0: model.board.display = False
        trainer.fit(model, data_fold)
        val_loss.append(float(model.board.data['val_loss'][-1].y))
        models.append(model)
    sorted_pairs = sorted(zip(val_loss, models), key=lambda x: x[0])
    best_pairs = sorted_pairs[:-3]
    remaining_losses, best_models = zip(*best_pairs)
    avg_loss = sum(remaining_losses) / len(remaining_losses)
    print(f'average validation log mse = {avg_loss}')
    return best_models

In [ ]:
#Actual traning, hyperparameters
hparams = {
        'num_outputs': 1,
        'num_hiddens_1': 256,
        'num_hiddens_2': 128,
        'num_hiddens_3': 64,
        'num_hiddens_4': 32,
        'num_hiddens_5': 16,
        'dropout_1': 0,
        'dropout_2': 0.03,
        'dropout_3': 0.06,
        'dropout_4': 0.09,
        'dropout_5': 0.12,
        'lr': 0.001,
        'wd': 0.0015,
        'activation': 'silu'
}
trainer = d2l.Trainer(max_epochs=70)
models = k_fold(trainer, data, k=10, hparams=hparams)

In [ ]:
#Prediction
preds = [model(torch.tensor(data.val.values.astype(float), dtype=torch.float32))
         for model in models]
# Taking exponentiation of predictions in the logarithm scale
ensemble_preds = torch.exp(torch.cat(preds, 1))
sorted_x, _ = torch.sort(ensemble_preds, 1)
trimmed_x = sorted_x[:, 2:-2]
finals = trimmed_x.mean(1)
submission = pd.DataFrame({'Id':data.raw_val.Id,
                           'SalePrice':finals.detach().numpy()})
submission.to_csv('submission.csv', index=False)